This notebook is to process SOW docs using LLM to extract action items and topics using claude-4.5 via aws bedrock with batch processing for efficiency. 

NOTE: this notebook requires the nlp_topics_env.yml environment.
NOTE: errors seen in this notebook are from testing after removal of api keys.

In [ ]:
import os
import re
import json
import time
import pandas as pd
from pathlib import Path
import boto3
import numpy as np
import tiktoken
import concurrent.futures as cf
from tqdm.auto import tqdm

# Regex Based Cleaning Functions for SOW Texts

These regex-based functions are used to clean Statement of Work (SOW) texts by removing administrative lines and normalizing the text for further processing through an LLM for action statement extraction. This step helps improve the quality of data fed into the LLM, and also reduces token usage by eliminating irrelevant content.

In [ ]:

_LEADIN_PATTERNS = [
    r"^as\s+directed\s+by[^\w]+fema[,\s\-:;]*",
    r"\bin\s+coordination\s+with\s+fema[,\s\-:;]*",
    r"\bper\s+fema\s+(request|direction)[,\s\-:;]*",
    r"\bas\s+requested\s+by\s+fema[,\s\-:;]*",
    r"\bin\s+(direct\s+)?support\s+(of|to)\s+fema[,\s\-:;]*"
]

def strip_leadins(text: str) -> str:
    t = " " + text.strip()
    for p in _LEADIN_PATTERNS:
        t = re.sub(p, " ", t, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", t).strip()



_ADMIN_LINE_PATTERNS = [
    r"agenc(?:y|ies)\s+will\s+be\s+reimbursed.*",
    r"eligible\s+expenses\s+pursuant\s+to.*",
    r"44\s*cfr\s*\d+.*",
    r"\bunliquidated\s+obligations?.*",
    r"\bulo\b.*",
    r"documentation\s+maintained\s+by\s+the\s+agency.*",
    r"matos?\s+may\s+be\s+issued\s+by\s*fema.*",
    r"ma\s+task\s+orders.*",
    r"shall\s+be\s+considered\s+for\s+closure.*",
    r"all\s+equipment\s+and\s+supply\s+purchases.*",
    r"prior\s+approval.*federal\s+approving\s+official.*"
]

_ACTION_VERBS = re.compile(
    r"\b(provide|deploy|assist|coordinate|conduct|deliver|assess|install|support|evaluate|operate|repair|monitor)\b",
    re.IGNORECASE
)

def strip_admin_lines(text: str) -> str:
    """Remove individual administrative/financial lines but keep any line with an action verb."""
    out_lines = []
    for line in re.split(r"[.\n]", text):
        ln = line.strip()
        if not ln:
            continue

        # keep if any valid action verb present
        if _ACTION_VERBS.search(ln):
            out_lines.append(ln)
            continue

        # drop if it matches admin/financial boilerplate
        if any(re.search(p, ln, flags=re.IGNORECASE) for p in _ADMIN_LINE_PATTERNS):
            continue

        # keep neutral lines that are not admin/financial
        out_lines.append(ln)

    return " ".join(out_lines).strip()

def normalize(text: str) -> str:
    t = re.sub(r"[^\w\s\-\/,().:]", " ", text)
    t = re.sub(r"\s+", " ", t)
    return t.strip().lower()


def clean_sow(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    t = strip_leadins(text)
    t = strip_admin_lines(t)   # SAFE: line-based cleaning, no block deletion
    return normalize(t)

DATA_PATH = Path(r"MissionAssignments.csv")
OUT_DIR   = Path(r"MA_topics_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, dtype=str)

df = df.dropna(subset=["statementOfWork"]).copy()
df["clean_statement"] = df["statementOfWork"].apply(clean_sow)

# OPTIONAL: prune extremely short cleaned SOWs (< ~10 words)
df = df[df["clean_statement"].str.split().str.len() >= 10].reset_index(drop=True)

df.to_csv(OUT_DIR / "mission_assignments_clean_safe.csv", index=False)

df.head()

,incidentId,incidentName,incidentType,disasterNumber,declarationType,declarationTitle,maId,maAmendNumber,actionId,maType,...,maFedCostSharePct,maSttCostShareAmount,maFedCostShareAmount,maPriority,assistanceRequested,statementOfWork,id,hash,lastRefresh,clean_statement
0,2024081901,R1_Severe Storm-Flooding_08182024,Severe Storm,3612,EM,"SEVERE STORMS, FLOODING, LANDSLIDES, AND MUDSL...",3612EMCTDOI-USGS01,0,476360,FOS,...,1.00,0.00,20000.00,High,USGS Field measurements of flood-water heights...,"As directed by and in coordination with FEMA, ...",ecbcf24e-7d9e-4ced-ae57-4871181703e1,497d57a5e4d071da1721548476b50b64a7321bdd,2024-08-27T18:24:53.439Z,as directed by and u s geological survey (usgs...
1,2024080901,Tropical Disturbance_TS_Ernesto,Tropical Storm,NaN,SU,NaN,PR24080901USDA-APH01,0,475083,FOS,...,1.00,0.00,29184.00,High,As directed by and in accordance with FEMA Reg...,"As directed by and in coordination with FEMA, ...",72dc30d3-d804-4005-897f-91395be284b2,e0e65997764dc0d660e5b941a18d2f18adabaa55,2024-08-27T18:24:53.439Z,as directed by and us dept of agriculture (usd...
2,2024080901,Tropical Disturbance_TS_Ernesto,Tropical Storm,NaN,SU,NaN,PR24080901DOT01,0,475067,FOS,...,1.00,0.00,25000.00,High,As directed by and in accordance with FEMA Reg...,"As directed by and in coordination with FEMA, ...",61a653ea-688c-4218-bef2-662e1c8b7fde,dbb5a30a3fbd2c96318ba12decb71e63ac072b76,2024-08-27T18:24:53.439Z,as directed by and dept of transportation (dot...
3,2024080901,Tropical Disturbance_TS_Ernesto,Tropical Storm,NaN,SU,NaN,PR24080901DOJ-ATF01,0,475098,FOS,...,1.00,0.00,73500.00,High,As directed by and in accordance with FEMA Reg...,"As directed by and in coordination with FEMA, ...",eeafed44-bdfa-45be-a2ec-6aeb4199043d,4f49b6e6d98634659a77d554c06b8fdeac4503cd,2024-08-27T18:24:53.439Z,as directed by and dept of justice (doj) will ...
4,2024080901,Tropical Disturbance_TS_Ernesto,Tropical Storm,NaN,SU,NaN,PR24080901DOE-OE01,0,475064,FOS,...,1.00,0.00,124822.00,High,As directed by and in accordance with FEMA Reg...,"As directed by and in coordination with FEMA, ...",bdb7935b-5df5-4124-87d8-ddd8feb4a8e3,35d63a525dff8c4a93b83d676d303b6089c0373a,2024-08-27T18:24:53.439Z,as directed by and dept of energy (doe) will p...


In [4]:
df.shape

(40438, 40)

# API and LLM configurations

This section contains configurations for interacting with AWS Bedrock to access claude-4.5 and select the correct inference models. This section also has some configuration settings for the batch size and how many batches to process in parallel. This is also where you select the column to send to the LLM for extraction. It will create 2 new columns in the dataframe: one for basic SOW topic and one for the extracted action statements. The basic is generated in the prompt but because the LLM is run on batches the topics are not guaranteed to be consistent across the returned results but they can give a general idea of what we are seeing in the SOWs.

NOTE: The output of this LLM is processed in the NLP exploration notebook and requires the code beyond this point to be run twice: once for clean_statement and once for statementOfWork column before saving the final output CSV.

In [ ]:
# credentials from lab environment
AWS_ACCESS_KEY_ID     = ""
AWS_SECRET_ACCESS_KEY = ""
AWS_SESSION_TOKEN     = ""

AWS_REGION      = "us-east-1"   # Region where Bedrock is enabled
MODEL_ID        = "anthropic.claude-haiku-4-5-20251001-v1:0"
INFERENCE_PROFILE_ARN = "arn:aws:bedrock:us-east-1:089484167146:inference-profile/us.anthropic.claude-3-5-haiku-20241022-v1:0"
COL             = "clean_statement"                      # input text column
ACTIONS_COL     = f"{COL}_actions_extracted"             # output: JSON list of action sentences
TOPIC_COL       = f"{COL}_topic"                         # output: one general topic string
CSV_PATH        = None                                    # optional: "data.csv" if you want to load from disk
OUT_PATH        = "actions_topics_bedrock_full_sow.csv"            # output CSV
TRUNCATE_CHARS  = 1200                                    # cap input length per row (Haiku has 200k ctx; 1200 keeps latency sane)
MAX_TOKENS      = 220                                     # cap on OUTPUT tokens per request (JSON only; keep modest)
TEMPERATURE     = 0.0
RETRIES         = 5
UTILIZATION     = 0.65                                    # fraction of 200k context to target per request (safety margin)
FIXED_OVERHEAD  = 700                                     # tokens for system, few-shots, JSON instructions
PER_ROW_OVERHEAD= 30                                      # approx tokens per row for ROW_ID + formatting
DEFAULT_BATCH   = 400                                     # used if tokenizer is unavailable
MAX_WORKERS     = 4                                       # parallel requests; for big batches keep 3–6
RESUME_ONLY_NEW = True                                    # skip rows that already have JSON in ACTIONS_COL

# Create Bedrock client using inline credentials
bedrock_ctl = boto3.client(
    "bedrock",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
)

bedrock_rt = boto3.client(
    "bedrock-runtime",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
)

print([m["modelId"] for m in bedrock_ctl.list_foundation_models()["modelSummaries"]][:10])

ClientError: An error occurred (UnrecognizedClientException) when calling the ListFoundationModels operation: The security token included in the request is invalid.

# Token Estimation and Batch Size Calculation

This section provides code to estimate the number tokens that each SOW text will use when sent to the LLM. This information is used to calculate a batch size that will allow us to send the largest batch we can at a time without exceeding the model's token limit. THis optimizes the number of calls needed to process all of the SOWs.

In [6]:
enc = tiktoken.get_encoding("o200k_base") 

def count_tokens_tiktoken(texts, max_chars=1200):
    return [len(enc.encode(str(t)[:max_chars])) for t in texts]

# Example with your df:
texts = df["clean_statement"].fillna("").astype(str)
token_counts = count_tokens_tiktoken(texts, max_chars=1200)

print("Avg:", np.mean(token_counts))
print("P90:", np.percentile(token_counts, 90))
print("Max:", np.max(token_counts))

Avg: 166.97922745932044
P90: 239.0
Max: 299


In [7]:
CONTEXT_LIMIT = 200_000
p90 = np.percentile(token_counts, 90) if len(token_counts) else 200
safe_batch = int(UTILIZATION * (CONTEXT_LIMIT - FIXED_OVERHEAD) / (p90 + PER_ROW_OVERHEAD))
if safe_batch <= 0:
    safe_batch = DEFAULT_BATCH
BATCH_SIZE = max(50, min(500, safe_batch))

print(f"Estimated tokens per row (p90): {int(p90)}")
print(f"Using BATCH_SIZE = {BATCH_SIZE}, MAX_WORKERS = {MAX_WORKERS}")


Estimated tokens per row (p90): 239
Using BATCH_SIZE = 481, MAX_WORKERS = 4


# LLM Promp Configuration

Here we define the prompt template that will be passed to the LLM with each batch of SOW texts. This also includes examples of desired outputs to help guide the LLM since we know that the SOWs can still vary widely in structure and content.

In [12]:
SYSTEM_POLICY = """
You are a meticulous data-cleaning assistant for FEMA Mission Assignments (SOW text).

Your job is to EXTRACT operational tasks and summarize them.

For each INPUT row:

1) ACTIONS
   Extract ONLY operational, observable, action-level tasks from the SOW text.
   - Each action must be a discrete imperative-style task (even if the SOW is not written as commands).
   - Include actions even if phrased generically (e.g., "Provide personnel support" IS VALID).
   - Keep tasks involving:
        • personnel support or staffing
        • assessments, inspections, monitoring
        • technical or analytical assistance
        • logistics, movement, staging
        • equipment deployment, resource support
        • coordination, planning, or communications activities
   - Exclude:
        • reimbursement/financial boilerplate
        • regulatory citations (CFR, Stafford Act, Privacy Act)
        • "funding will be provided..." lines
        • purely administrative recordkeeping/audit/ULO text, unless it clearly describes an operational activity
   - If a SOW contains only generic operational support (e.g., "provide personnel"),
     STILL include it as at least 1 action.
   - Only return an empty actions list if the SOW truly contains no operational
     activity at all and is purely financial, regulatory, or administrative.

2) TOPIC
   Provide ONE concise topic (1–5 words), lowercase, no period.
   - Summarize the type of operational support captured by the actions.
   - Examples: "logistics support", "personnel deployment", "damage assessment",
               "technical assistance", "shelter operations".
   - If there is truly no operational content and actions is an empty list,
     use a generic topic like "administrative / financial".

Return ONLY a single JSON object of the form:
{ "row_id": {"actions": [...], "topic": "..."}, ... }

Do NOT fabricate actions that are not implied or stated in the SOW.
Do NOT merge or reuse actions from other rows.
Do NOT add any text before or after the JSON.
"""

# examples for few-shot prompting

FEW_SHOT_EXAMPLES = [
    {
        "input": "Provide staffing for the regional logistics staging area and coordinate distribution of water and tarps to impacted counties. Reimbursement will follow per 2 CFR 200.",
        "actions": [
            "Provide staffing for the regional logistics staging area.",
            "Coordinate distribution of water and tarps to impacted counties."
        ],
        "topic": "logistics support"
    },
    {
        "input": "Assess shelter capacity and operate two temporary shelters for 14 days. In accordance with the Privacy Act, records will be maintained.",
        "actions": [
            "Assess shelter capacity.",
            "Operate two temporary shelters for 14 days."
        ],
        "topic": "shelter operations"
    },
    {
        "input": "EPA will provide appropriate personnel to support disaster operations at RRCC, IOF, and JFO. Personnel will assist rapid needs assessment teams as requested.",
        "actions": [
            "Provide personnel to support disaster operations at RRCC, IOF, and JFO.",
            "Assist rapid needs assessment teams as requested."
        ],
        "topic": "personnel support"
    },
    {
        "input": "USGS will conduct field inspections, monitor ground movement, and provide technical assistance to state geologists. Funding provisions apply per Stafford Act.",
        "actions": [
            "Conduct field inspections.",
            "Monitor ground movement.",
            "Provide technical assistance to state geologists."
        ],
        "topic": "geological assessment"
    },
    {
        "input": "FEMA will coordinate delivery of generators to critical facilities and support power restoration efforts with technical advisors.",
        "actions": [
            "Coordinate delivery of generators to critical facilities.",
            "Support power restoration efforts with technical advisors."
        ],
        "topic": "generator support"
    },
    {
        "input": "Provide personnel to staff the Joint Information Center and support public communication during the response.",
        "actions": [
            "Provide personnel to staff the Joint Information Center.",
            "Support public communication during the response."
        ],
        "topic": "public information support"
    },
    {
        "input": "This mission assignment provides funding for salary, travel, and overtime costs associated with personnel previously deployed under prior mission assignments. Funds will be used to reimburse eligible expenses pursuant to the Stafford Act and applicable FEMA policies.",
        "actions": [],
        "topic": "administrative / financial"
    }
]

# building of the LLM input prompt for batches
def _truncate(text, max_chars=1200) -> str:
    """Collapse whitespace and hard-truncate long SOW text."""
    if text is None:
        return ""
    t = re.sub(r"\s+", " ", str(text)).strip()
    if len(t) > max_chars:
        t = t[:max_chars]
    return t

def build_batch_prompt(pairs):
    """
    Build a single prompt string for a batch of rows.

    Parameters
    ----------
    pairs : list of (row_id_str, text)
        row_id_str should be the exact key you want in the JSON result.
        text is the (cleaned or raw) SOW text for that row.

    Returns
    -------
    prompt : str
        Full prompt including SYSTEM_POLICY, few-shot examples, and batch rows.
    """
    lines = []

    lines.append(SYSTEM_POLICY.strip())
    lines.append("\nYou will now process multiple FEMA SOW texts in one batch.\n")
    lines.append(
        "You will be given multiple FEMA Mission Assignment SOW texts, each with a row_id.\n"
        "For EACH row_id, extract:\n"
        "- actions: a list of operational action-level tasks (see rules above), and\n"
        "- topic: one short general topic.\n"
        "Return ONLY ONE JSON object mapping each row_id to its actions and topic."
    )
    lines.append("\nEXAMPLES:\n")
    for i, ex in enumerate(FEW_SHOT_EXAMPLES, start=1):
        ex_id = f"example_{i}"
        lines.append(f"INPUT (row_id = \"{ex_id}\"):")
        lines.append(ex["input"])
        lines.append("OUTPUT JSON:")
        example_obj = {
            ex_id: {
                "actions": ex["actions"],
                "topic": ex["topic"],
            }
        }
        lines.append(json.dumps(example_obj, ensure_ascii=False, indent=2))
        lines.append("")  # blank line between examples

    lines.append("\nNOW PROCESS THE FOLLOWING ROWS.\n")
    for rid, txt in pairs:
        lines.append(f'ROW_ID: "{rid}"')
        lines.append("TEXT:")
        lines.append(_truncate(txt))
        lines.append("---")  # simple separator
    row_ids = [str(rid) for rid, _ in pairs]
    lines.append(
        "\nReturn ONLY a single JSON object with this structure:\n"
        "{\n"
        '  \"row_id1\": {\"actions\": [\"...\"], \"topic\": \"...\"},\n'
        '  \"row_id2\": {\"actions\": [\"...\"], \"topic\": \"...\"},\n'
        "  ...\n"
        "}\n"
        f"The keys MUST be the exact row_id values: {row_ids}.\n"
        "Do not include any keys for rows that were not provided.\n"
        "Do not add any text before or after the JSON."
    )

    return "\n".join(lines)


# Response Processing Functions

This section provides functions for parsing the LLM response and extracting the list of action statements and the topics for each SOW. The response is a json object that maps each SOW row ID to its extraced information.

In [13]:
def parse_json_obj(raw):
    try:
        return json.loads(raw)
    except Exception:
        m = re.search(r"\{.*\}", raw, re.S)
        if m:
            try: return json.loads(m.group(0))
            except Exception: pass
    return None

def clean_actions(lst):
    out, seen = [], set()
    for x in lst or []:
        if not isinstance(x, str): continue
        s = re.sub(r"\s+", " ", x).strip()
        if not s: continue
        if not s.endswith("."): s += "."
        k = s.lower()
        if k not in seen:
            seen.add(k)
            out.append(s)
    return out

def clean_topic(s):
    if not isinstance(s, str): return ""
    t = re.sub(r"\s+", " ", s).strip().lower()
    t = re.sub(r"[.]+$", "", t)   # remove trailing periods
    # keep short topic
    return t[:80]

def bedrock_generate(prompt, system=None, max_tokens=220, temperature=0.0, debug=False):
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "messages": [
            {"role": "user", "content": [{"type": "text", "text": prompt}]}
        ],
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    if system:
        body["system"] = [{"type": "text", "text": system}]

    # --- invoke via inference PROFILE (no on-demand) ---
    # (Works with older SDKs by passing the profile ARN as modelId)
    resp = bedrock_rt.invoke_model(
        modelId=INFERENCE_PROFILE_ARN,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json",
    )

    # decode and parse
    raw = resp["body"].read().decode("utf-8")
    try:
        out = json.loads(raw)
    except Exception:
        out = {}

    # Anthropic-on-Bedrock returns: {"type":"message","content":[{"type":"text","text":"..."}], ...}
    text = ""
    if isinstance(out, dict) and "content" in out:
        parts = [c.get("text", "") for c in out.get("content", []) if c.get("type") == "text"]
        text = "\n".join(p for p in parts if p).strip()
    else:
        # fall back to raw text if provider already returned plain text
        text = raw

    if debug:
        print("---- REQUEST BODY (trunc) ----")
        print(json.dumps(body, indent=2)[:800])
        print("---- RAW RESPONSE (trunc) ----")
        print(raw[:1000])
        print("---- PARSED TEXT (trunc) ----")
        print(text[:500])

    return text

def process_batch(pairs):
    """Returns dict {row_id: {"actions":[...], "topic":"..."}} with retries; per-row fallback on last retry."""
    prompt = build_batch_prompt(pairs)
    delay = 1.0
    for attempt in range(1, RETRIES+1):
        try:
            raw = bedrock_generate(prompt, system=SYSTEM_POLICY, max_tokens=MAX_TOKENS, temperature=TEMPERATURE)
            obj = parse_json_obj(raw)
            expected = set(rid for rid,_ in pairs)
            if isinstance(obj, dict) and expected.issubset(obj.keys()):
                out = {}
                for rid, _ in pairs:
                    val = obj.get(rid, {}) or {}
                    acts = clean_actions(val.get("actions", []))
                    topic = clean_topic(val.get("topic", ""))
                    out[rid] = {"actions": acts, "topic": topic}
                return out
        except Exception:
            pass
        if attempt < RETRIES:
            time.sleep(delay); delay *= 2.0

    # Last resort: process each row individually to salvage the batch
    results = {}
    for rid, txt in pairs:
        sp = build_batch_prompt([(rid, txt)])
        try:
            raw1 = bedrock_generate(sp, system=SYSTEM_POLICY, max_tokens=MAX_TOKENS, temperature=TEMPERATURE)
            o1 = parse_json_obj(raw1) or {}
            v = o1.get(rid, {}) if isinstance(o1, dict) else {}
            results[rid] = {
                "actions": clean_actions(v.get("actions", [])),
                "topic": clean_topic(v.get("topic", ""))
            }
        except Exception:
            results[rid] = {"actions": [], "topic": ""}
    return results

def bedrock_generate_ip(prompt, system=None, max_tokens=500, temperature=0.0, debug=True):
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "messages": [{"role": "user", "content": [{"type": "text", "text": prompt}]}],
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    if system:
        body["system"] = [{"type": "text", "text": system}]

    if debug:
        print("\n==== REQUEST BODY PREVIEW ====")
        try:
            print(json.dumps(body, indent=2)[:1200])
        except Exception:
            print(str(body)[:1200])
        print("==============================")

    resp = bedrock_rt.invoke_model(
        modelId=INFERENCE_PROFILE_ARN,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json",
    )

    raw = resp["body"].read().decode("utf-8")

    if debug:
        print("\n==== RAW RESPONSE PREVIEW ====")
        print(raw[:1200])
        print("==============================\n")

    # Keep the rest of your original function logic intact
    try:
        out = json.loads(raw)
        if isinstance(out, dict) and "content" in out:
            parts = [c.get("text", "") for c in out["content"] if c.get("type") == "text"]
            text = "\n".join(p for p in parts if p).strip()
        else:
            text = raw
    except Exception:
        text = raw

    if debug:
        print("---- PARSED TEXT (trunc) ----")
        print(text[:500])

    return text

# Batch Building

In [14]:
existing_actions = df[ACTIONS_COL].astype(str).tolist() if ACTIONS_COL in df.columns else [None]*len(df)
existing_topic   = df[TOPIC_COL].astype(str).tolist()   if TOPIC_COL   in df.columns else [None]*len(df)

def row_done(i):
    if not RESUME_ONLY_NEW: return False
    a = existing_actions[i]
    t = existing_topic[i] if i < len(existing_topic) else None
    ok_a = isinstance(a, str) and a.strip().startswith("[")
    ok_t = isinstance(t, str) and len(t.strip()) > 0
    return ok_a and ok_t

df = df.reset_index(drop=True)
texts = df[COL].fillna("").astype(str).tolist()

pending_idx = [i for i in range(len(df)) if not row_done(i)]
batches = []
for i in range(0, len(pending_idx), BATCH_SIZE):
    idxs = pending_idx[i:i+BATCH_SIZE]
    pairs = [(str(j), texts[j]) for j in idxs]
    batches.append(pairs)

# Dry run test of LLM call and input

In [15]:
DRY_RUN = True  # set False when ready to run

if DRY_RUN:
    for b in batches:
        print(f"Batch size={len(b)}")
        for rid, txt in b[:3]:  # show first 3 samples
            print(f"\n[Row {rid}]\n{txt[:400]}...")
    raise SystemExit("Dry-run complete — nothing sent to Bedrock.")

Batch size=481

[Row 0]
as directed by and u s geological survey (usgs) will provide advance support, real-time field measurements, and daily reporting of water heights in direct support and for situational awareness of fema disaster operations for a high-water or flood event usgs services may include, but are not limited to, the following in direct support of response and recovery operations: - field measurements of flo...

[Row 1]
as directed by and us dept of agriculture (usda) will provide appropriate personnel to the rrcc, iof, jfo, or other fema teams or facilities in support of disaster operations requirements from fema: 1 2 supporting documentation is required for reimbursement work that falls within the statutory authority of the performing federal agency is not eligible for fema reimbursement 3 if approved, document...

[Row 2]
as directed by and dept of transportation (dot) will provide appropriate personnel to rrcc, iof, jfo, or others teams and facilities as requested fund

SystemExit: Dry-run complete — nothing sent to Bedrock.

f:\anaconda3_newest\envs\nlp_topics_env\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# Single SOW test input and output

In [ ]:
def build_batch_prompt_one(rid, txt):
    return f'''
Extract action-level tasks and a single topic for each row independently.
Return ONLY a single JSON object mapping row_id -> {{"actions": [...], "topic": "..."}}.

ROW_ID: "{rid}"
TEXT:
{_truncate(txt)}

RETURN JSON:
'''.strip()

row_id = 29  # choose a row index that exists in df
txt = "the US Army Corps of Engineers (USACE) will provide appropriate personnel to the RRCC, IOF, JFO, or other facilities in support of disaster operations. Support may include, but is not limited to, the following: USACE support to RRCC USACE support to National and/or Regional IMAT USACE modeling and/or National Hurricane Program support USACE Local Government Liaison(s) USACE support to ESF #15 External Affairs USACE SME(s) Infrastructure Systems Recovery Support Function (ISRSF) Field Coordinator (under NDRF) Contract or OFA audit support to ensure costs incurred under USACE MAs meet all Stafford Act and regulatory requirements This mission assignment may also fund other site specific personnel and Non mission specific administrative and management support at a Recovery Field Office (RFO) or other USACE node locations. Those costs are not considered in this initial estimate. Costs would vary based on the magnitude and duration of the response and recovery operations as authorized by FEMA."
probe_prompt = build_batch_prompt_one(str(row_id), txt)

resp_text = bedrock_generate_ip(probe_prompt, system=SYSTEM_POLICY, max_tokens=500, temperature=0.0, debug=True)
obj = parse_json_obj(resp_text)
print("\nPARSED JSON:\n", json.dumps(obj, indent=2, ensure_ascii=False))


==== REQUEST BODY PREVIEW ====
{
  "anthropic_version": "bedrock-2023-05-31",
  "messages": [
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "Extract action-level tasks and a single topic for each row independently.\nReturn ONLY a single JSON object mapping row_id -> {\"actions\": [...], \"topic\": \"...\"}.\n\nROW_ID: \"29\"\nTEXT:\nthe US Army Corps of Engineers (USACE) will provide appropriate personnel to the RRCC, IOF, JFO, or other facilities in support of disaster operations. Support may include, but is not limited to, the following: USACE support to RRCC USACE support to National and/or Regional IMAT USACE modeling and/or National Hurricane Program support USACE Local Government Liaison(s) USACE support to ESF #15 External Affairs USACE SME(s) Infrastructure Systems Recovery Support Function (ISRSF) Field Coordinator (under NDRF) Contract or OFA audit support to ensure costs incurred under USACE MAs meet all Stafford Act an

# Submnit LLM calls in batches and process results 

In [ ]:
results_actions = existing_actions[:] if existing_actions else [None]*len(df)
results_topic   = existing_topic[:]   if existing_topic   else [None]*len(df)

if batches:
    with cf.ThreadPoolExecutor(max_workers=8) as ex:
        futures = [ex.submit(process_batch, b) for b in batches]
        for fut in tqdm(cf.as_completed(futures), total=len(futures), desc=f"Bedrock batches (size={BATCH_SIZE})"):
            out = fut.result()
            for rid, obj in (out or {}).items():
                i = int(rid)
                results_actions[i] = json.dumps(obj.get("actions", []), ensure_ascii=False)
                results_topic[i]   = obj.get("topic", "")

df[ACTIONS_COL] = results_actions
df[TOPIC_COL]   = results_topic

Bedrock batches (size=481): 100%|██████████| 85/85 [7:28:29<00:00, 316.59s/it]    


Done. Filled rows: 40438 / 40438 | Saved -> actions_topics_bedrock_cleaned_statement_v3.csv


# Save New Dataframe with Extracted Actions and Topics

In [ ]:
if OUT_PATH:
    df.to_csv(OUT_PATH, index=False)

print("Done. Filled rows:",
      sum(isinstance(x,str) and x.strip().startswith('[') for x in df[ACTIONS_COL]), "/", len(df),
      "| Saved ->", OUT_PATH)